In [1]:
import pandas as pd
import numpy as np
from numpy.core.records import fromarrays
from scipy.io import savemat

In [27]:
choice = 3  #start from 0
# devices = ['nfet_03v3', 'pfet_03v3']
devices = ['nfet_03v3']

# widths used for characterization
w = np.array([5.0, 4.0, 5.0, 2])
nfing = np.array([1, 4, 1, 6])

In [28]:
# read ngspice data
df_raw = pd.read_csv('./simulation/techsweep_'+devices[0]+'_w_'+str(int(w[choice]))+'um'+'_nf_'+str(nfing[choice])+'.txt', sep='\s+')
par_names = df_raw.columns.to_list()
fet_name = par_names[1].split('[')[0]

# remove unwanted columns and rename for readability
df = df_raw.drop(['frequency', 'frequency.1'], axis=1)
df = df.apply(pd.to_numeric)
df.columns = df.columns.str.replace(fet_name, '')
df.columns = df.columns.str.replace(fet_name[1:], '')
df.columns = df.columns.str.replace('[dc]', '')
df.columns = df.columns.str.replace('onoise..', 'n')
df.columns = df.columns.str.removeprefix('@')
df.columns = df.columns.str.removeprefix('[')
df.columns = df.columns.str.removesuffix(']')
df

,capbd,capbs,cdd,cgb,cgd,cgg,cgs,css,gds,gm,gmbs,id,l,vth,vb,vd,vg,n1overf,nid
0,1.544000e-15,1.764000e-15,3.829000e-19,-7.041000e-16,-3.663000e-19,7.048000e-16,-2.828000e-19,2.521000e-20,4.661000e-12,-9.614000e-35,-2.898000e-35,0.000000e+00,2.800000e-07,0.6307,0.0,0.000,0.0,0.000000e+00,3.276000e-16
1,1.439000e-15,1.650000e-15,1.623000e-19,-6.352000e-16,-1.559000e-19,6.354000e-16,-3.963000e-20,9.827000e-21,9.586000e-13,0.000000e+00,0.000000e+00,0.000000e+00,2.800000e-07,0.6720,-0.2,0.000,0.0,0.000000e+00,1.485000e-16
2,1.359000e-15,1.564000e-15,7.644000e-20,-5.830000e-16,-7.361000e-20,5.830000e-16,7.869000e-20,4.353000e-21,2.366000e-13,0.000000e+00,-3.340000e-32,0.000000e+00,2.800000e-07,0.7081,-0.4,0.000,0.0,0.000000e+00,7.373000e-17
3,1.529000e-15,1.764000e-15,5.482000e-20,-7.041000e-16,-4.746000e-20,7.047000e-16,-5.368000e-19,1.634000e-19,2.446000e-12,2.438000e-12,7.364000e-13,8.734000e-14,2.800000e-07,0.6302,0.0,0.025,0.0,3.387000e-17,2.873000e-16
4,1.428000e-15,1.650000e-15,2.317000e-20,-6.352000e-16,-1.998000e-20,6.353000e-16,-1.483000e-19,6.869000e-20,5.066000e-13,5.165000e-13,1.333000e-13,1.802000e-14,2.800000e-07,0.6714,-0.2,0.025,0.0,7.142000e-18,1.303000e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636799,9.300000e-16,1.650000e-15,5.017000e-18,3.628000e-16,-4.234000e-18,1.835000e-14,-1.871000e-14,1.298000e-14,7.673000e-07,1.238000e-04,5.186000e-05,1.945000e-04,3.000000e-06,0.7104,-0.2,3.275,3.3,1.815000e-09,1.335000e-12
636800,9.177000e-16,1.564000e-15,4.877000e-18,7.982000e-17,-4.234000e-18,1.819000e-14,-1.826000e-14,1.268000e-14,7.076000e-07,1.197000e-04,4.537000e-05,1.848000e-04,3.000000e-06,0.7779,-0.4,3.275,3.3,1.744000e-09,1.317000e-12
636801,9.413000e-16,1.763000e-15,4.900000e-18,8.022000e-16,-3.996000e-18,1.854000e-14,-1.934000e-14,1.337000e-14,8.244000e-07,1.289000e-04,6.081000e-05,2.057000e-04,3.000000e-06,0.6362,0.0,3.300,3.3,1.898000e-09,1.355000e-12
636802,9.284000e-16,1.650000e-15,4.808000e-18,3.628000e-16,-4.058000e-18,1.835000e-14,-1.871000e-14,1.298000e-14,7.540000e-07,1.238000e-04,5.187000e-05,1.945000e-04,3.000000e-06,0.7104,-0.2,3.300,3.3,1.816000e-09,1.335000e-12


In [29]:
# sweep variable vectors
l =   np.round(np.unique(df['l'])*1e6, 2)
vgs = np.unique(df['vg'])
vds = np.unique(df['vd'])
vsb = np.unique(-df['vb'])

In [30]:
# ngspice sweep order is l, vgs, vds, vsb
dims = [len(l), len(vgs), len(vds), len(vsb)]
id = np.reshape(df['id'].values, dims)
vt = np.reshape(df['vth'].values, dims)
gm = np.reshape(df['gm'].values, dims)
gmb = np.reshape(df['gmbs'].values, dims)
gds = np.reshape(df['gds'].values, dims)
cgg = np.reshape(df['cgg'].values, dims) \
#      + np.reshape(df['cgdo'].values, dims) + np.reshape(df['cgso'].values, dims)
cgb = -np.reshape(df['cgb'].values, dims)
cgd = -np.reshape(df['cgd'].values, dims) \
#      + np.reshape(df['cgdo'].values, dims)
cgs = -np.reshape(df['cgs'].values, dims) \
#      + np.reshape(df['cgso'].values, dims)
cdd = np.reshape(df['cdd'].values, dims) \
      + np.reshape(df['capbd'].values, dims) \
#      + np.reshape(df['cgdo'].values, dims)
css = np.reshape(df['css'].values, dims) \
      + np.reshape(df['capbs'].values, dims) \
#      + np.reshape(df['cgso'].values, dims)
sth = np.reshape(df['nid'].values, dims)**2
sfl = np.reshape(df['n1overf'].values, dims)**2


In [35]:
dic = {
  "INFO": "GlobalFoundries, 180nm MCU CMOS , BSIM4",
  "CORNER": "NOM",
  "TEMP": 300.0,
  "VGS": vgs,
  "VDS": vds,
  "VSB": vsb,
  "L": l,
  "W": w[choice],
  "NFING": nfing[choice],
  "ID": id,
  "VT": vt,
  "GM": gm,
  "GMB": gmb,
  "GDS": gds,
  "CGG": cgg,
  "CGB": cgb,
  "CGD": cgd,
  "CGS": cgs,
  "CDD": cdd,
  "CSS": css,
  "STH": sth,
  "SFL": sfl
}
savemat('./simulation/'+devices[0]+'_w_'+str(int(w[choice]))+'um'+'_nf_'+str(nfing[choice])+'.mat', {devices[0]: dic})